# Week 31, Day 2: The exponential integrate-and-fire: where the spike begins

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w031/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The exponential foot of the sodium current

Compute the HH sodium current `INa` for V from -75 to -60 mV with m = m∞(V) and h fixed at h∞(-65) = 0.596, fit a straight line to ln(-INa) against V, and convert its slope into a slope factor `Delta_T` = 1 / slope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def minf(V):
    V = V + 1e-7
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    return am / (am + bm)

h0 = 0.596
V = np.arange(-75, -59.9, 0.5)
INa = ...
slope = np.polyfit(V, np.log(-INa), 1)[0]
Delta_T = ...
print(f"sodium current grows e-fold every {Delta_T:.2f} mV below threshold")
plt.semilogy(V, -INa, "o"); plt.xlabel("V (mV)"); plt.ylabel("-I_Na (uA/cm^2)"); plt.show()

In [ ]:
import numpy as np
def test_current():
    "INa is the HH sodium current with m at its steady state and h fixed at its resting value"
    assert np.allclose(INa, 120 * minf(V) ** 3 * h0 * (V - 50))

def test_slope():
    "Delta_T from the fit of ln(-INa) on V between -75 and -60 mV is between 2 and 6 mV"
    assert 2 < Delta_T < 6

test_current()
test_slope()
print("All checks passed")

### Exercise: Checking the rheobase

For an EIF neuron with C = 200 pF, gL = 10 nS, EL = -65 mV, VT = -52 mV and ΔT = 1.5 mV, compute the rheobase from the formula, then find it by bisection with `fires(I)`, which simulates 500 ms of constant current and reports whether the voltage ran away.

In [ ]:
import math
C, gL, EL, VT, DT = 200.0, 10.0, -65.0, -52.0, 1.5

def fires(I, T=500.0, dt=0.05):
    V = EL
    for k in range(int(T / dt)):
        V += dt * (-gL * (V - EL) + gL * DT * math.exp(min((V - VT) / DT, 30)) + I) / C
        if V > 0:
            return True
    return False

I_rh_formula = ...
lo, hi = 0.0, 300.0
for _ in range(20):
    mid = (lo + hi) / 2
    ...
I_rh_sim = hi
print(f"rheobase: formula {I_rh_formula:.1f} pA, simulation {I_rh_sim:.1f} pA;  an LIF with threshold VT would need {gL * (VT - EL):.0f} pA")

In [ ]:
import numpy as np
def test_formula():
    "I_rh_formula is gL (VT - EL - DeltaT)"
    assert abs(I_rh_formula - 10 * (-52 + 65 - 1.5)) < 1e-9

def test_bisection():
    "the simulated rheobase agrees with the formula within 2 pA"
    assert abs(I_rh_sim - I_rh_formula) < 2

test_formula()
test_bisection()
print("All checks passed")

### Exercise: Reading the model from a recording

A noise-driven EIF 'recording' (5 s) is simulated with known parameters. Complete `var_for(C)`, the variance of I - C dV/dt for samples between -66 and -64 mV; choose `C_est` on the grid `Cs`; then compute the binned F(V) = (C dV/dt - I)/C and fit it with the EIF form by grid search over ΔT and VT, solving for 1/τ and EL/τ by least squares.

In [ ]:
import numpy as np
r = np.random.default_rng(1)
dt, T = 0.05, 5000.0
n = int(T / dt)
I = np.empty(n); x = 0.0; a = np.exp(-dt / 5.0); xi = r.normal(size=n)
for k in range(n):                               # an Ornstein-Uhlenbeck current, as in Week 24
    x = a * x + np.sqrt(1 - a * a) * xi[k]; I[k] = 150 + 120 * x

V = np.empty(n); v, last, spikes = -65.0, -1e9, []
for k in range(n):                               # the ground-truth EIF neuron
    if k * dt - last < 2.0:
        v = -60.0; V[k] = v; continue
    v += dt * (-10 * (v + 65) + 10 * 1.5 * np.exp(min((v + 52) / 1.5, 20)) + I[k]) / 200.0
    if v > -30:
        spikes.append(k); last = k * dt; V[k] = 20.0; v = -60.0; continue
    V[k] = v

dVdt = np.append((V[1:] - V[:-1]) / dt, 0.0)
ok = np.ones(n, bool)
for s in spikes:
    ok[max(0, s - 20):s + 100] = False           # drop 1 ms before to 5 ms after each spike
ok &= V < -42

def var_for(C):
    ...

Cs = np.arange(100, 300, 2.0)
C_est = Cs[int(np.argmin([var_for(c) for c in Cs]))]
F = ((C_est * dVdt - I) / C_est)[ok]; Vm = V[ok]
edges = np.arange(-72, -43.9, 1.0); centres = (edges[1:] + edges[:-1]) / 2
counts = np.array([np.sum((Vm >= a_) & (Vm < b_)) for a_, b_ in zip(edges[:-1], edges[1:])])
keep = counts > 30                               # drop bins with too few samples
centres = centres[keep]
Fm = np.array([F[(Vm >= a_) & (Vm < b_)].mean() for a_, b_ in zip(edges[:-1][keep], edges[1:][keep])])
best = (np.inf, None, None, None)
for DT in np.arange(0.5, 4.0, 0.05):
    for VT in np.arange(-58, -44, 0.1):
        X = np.column_stack([-centres + DT * np.exp((centres - VT) / DT), np.ones(centres.size)])
        coef = np.linalg.lstsq(X, Fm, rcond=None)[0]
        err = np.sum((X @ coef - Fm) ** 2)
        if err < best[0]:
            best = (err, DT, VT, coef)
_, DT_fit, VT_fit, coef = best
tau_fit = 1 / coef[0]; EL_fit = coef[1] * tau_fit
print(f"{len(spikes) / 5:.1f} Hz;  C = {C_est:.0f} pF, tau = {tau_fit:.1f} ms, EL = {EL_fit:.1f} mV, VT = {VT_fit:.1f} mV, DeltaT = {DT_fit:.2f} mV")
print("truth: C = 200 pF, tau = 20 ms, EL = -65 mV, VT = -52 mV, DeltaT = 1.5 mV")

In [ ]:
import numpy as np
def test_capacitance():
    "C_est minimises the variance of I - C dV/dt between -66 and -64 mV, and is within 5% of 200 pF"
    assert abs(C_est - Cs[np.argmin([var_for(c) for c in Cs])]) < 1e-9 and abs(C_est - 200) < 10

def test_params():
    "the fitted EL is within 1 mV, VT within 1.5 mV and DeltaT within 0.6 mV of the truth"
    assert abs(EL_fit + 65) < 1 and abs(VT_fit + 52) < 1.5 and abs(DT_fit - 1.5) < 0.6

test_capacitance()
test_params()
print("All checks passed")